This notebook is tuned for a single **NVIDIA A100 40GB** (SXM4). For the original T4 / free-Colab version, see the upstream Unsloth notebook.
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

# Goal: Fine-tune Gemma 4 on the `direct_debit_explainer` API with RLVR

A production `direct_debit_explainer` API is served by gemini-2.5-pro today. Given a customer's `account_context` and the `latest_dd_change`, it returns a list of `TriggerExplanation`s — one per relevant reason the direct debit changed, drawn from a closed 7-value `Trigger` enum.

Online evaluation of that API flagged a material failure rate (up to ~35% on a representative 7-day window) across three categories: incorrectly identified previous-DD amounts, hallucinated tariff names or rate percentages, and inappropriate use of "underpayment" language. All three are **verifiable** — you can check them against the input.

In this notebook we GRPO-fine-tune Gemma 4 against the same I/O contract using:
- a synthetic `DDExplainerPromptInput` dataset with ground-truth trigger labels (from `dd_explainer.py`),
- reward functions that encode both the general domain rules (trigger detection via `detect_triggers`) and the three specific failure categories from the production error analysis.

# Installation
We'll be using [Unsloth](https://github.com/unslothai/unsloth) to do RL on Gemma 4. Unsloth saves 70% VRAM usage and makes reinforcement learning 2 to 6x faster.

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    # Gemma 4 requires transformers >= 5.5.0 — do NOT pin to 4.x here
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
# Gemma 4 requires transformers >= 5.5.0
!uv pip install --upgrade --no-deps "transformers>=5.5.0" tokenizers "trl>=0.28.0" unsloth unsloth_zoo

In [2]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

In [ ]:
%%capture
# Flash Attention 2 — if Unsloth reports FA2 as broken and falls back to xFormers,
# recompiling against the installed CUDA + torch fixes it. Takes 2-10 min on first
# run; cached afterwards. Skip this cell if FA2 already works.
!uv pip install --no-build-isolation "flash-attn>=2.7.0"

In [3]:
# Redirect heavy caches + checkpoints to the /workspace NVMe volume (195GB)
# instead of the overlay root. MUST run before any HF / torch import.
import os
WORKSPACE = "/workspace/gemma4_rl"
os.makedirs(f"{WORKSPACE}/hf_cache", exist_ok=True)
os.makedirs(f"{WORKSPACE}/torch_cache", exist_ok=True)
os.makedirs(f"{WORKSPACE}/outputs", exist_ok=True)

os.environ["HF_HOME"]           = f"{WORKSPACE}/hf_cache"
os.environ["HF_DATASETS_CACHE"] = f"{WORKSPACE}/hf_cache/datasets"
os.environ["TORCH_HOME"]        = f"{WORKSPACE}/torch_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # faster model downloads
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{WORKSPACE}/torch_cache/inductor"  # persist compiled Triton kernels on NVMe

### Unsloth

In [ ]:
from unsloth import FastModel
import torch
# DD explainer prompt is ~3.1k tokens (domain knowledge + input JSON); bump from 4096 so completions have room.
max_seq_length = 6144
lora_rank = 64         # A100 40GB easily fits 64; 128 also viable but slower.

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
] # More models at https://huggingface.co/unsloth

# Switched to FastModel (unified text/vision/audio loader) + E4B-it for stronger priors.
# Drop back to E2B-it if you OOM: saves ~2 GB of resident VRAM.
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,   # A100 40GB has room for 16-bit LoRA on E4B.
    full_finetuning = False,
)

To do efficient RL, we will use [LoRA](https://arxiv.org/abs/2106.09685), which allows us to only add 1 to 5% of extra weights to the model for finetuning purposes. This allows us to save memory usage by over 60%, and yet it retains good accuracy.

In [ ]:
# Text-only LoRA: skip vision/audio towers. The DD task has no images/audio, so
# attaching LoRA adapters there just wastes parameters.
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,   # attention LoRA is the biggest GRPO lever
    finetune_mlp_modules       = True,
    r = lora_rank,
    lora_alpha = lora_rank * 2,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = False,  # KV cache off is ~5-10× slower on Gemma 4 generation
    random_state = 3407,
)

# Synthetic DD explainer dataset

The input schema (`DDExplainerPromptInput`), output schema (`DirectDebitExplainerResponse`), domain knowledge, and scenario-first synthetic data generator all live in `dd_explainer.py`. Each generated example picks a target set of `Trigger`s and constructs an `AccountContext` whose data satisfies the domain rules for exactly that set — giving us a ground-truth label for every training example.

In [6]:
import sys
sys.path.insert(0, "/workspace/gemma4_rl")

from dd_explainer_data_generator import (
    DDExplainerPromptInput,
    DirectDebitExplainerResponse,
    Trigger,
    TriggerExplanation,
    detect_triggers,
    generate_dd_example,
    build_dataset,
    build_chat_messages,
    SYSTEM_PROMPT,
    DOMAIN_KNOWLEDGE,
)

print("Triggers in the closed enum:")
for t in Trigger:
    print(f"  - {t.value}")

Triggers in the closed enum:
  - Manual reduction
  - Exemption Expiry
  - Change in usage
  - Change in unit rates
  - Missed/bounced DD payments
  - First DD review since account start
  - No triggers identified


Generate one example and inspect it — a single `DDExplainerPromptInput` plus the target triggers the generator was asked to produce. `detect_triggers` is the rule-based oracle; when we call it on the freshly generated input, it should recover the exact target set.

In [7]:
import random, json

rng = random.Random(7)
target = {Trigger.Change_in_usage, Trigger.Missed_bounced_DD_payments}
example = generate_dd_example(target, rng)

print(f"Target triggers: {sorted(t.value for t in target)}")
print(f"Oracle recovered: {sorted(t.value for t in detect_triggers(example))}")
print()
print("latest_dd_change:")
print(json.dumps(example.latest_dd_change.model_dump(mode='json'), indent=2, default=str))

Target triggers: ['Change in usage', 'Missed/bounced DD payments']
Oracle recovered: ['Change in usage', 'Missed/bounced DD payments']

latest_dd_change:
{
  "datetime_from": "2026-02-13T08:00:00",
  "datetime_to": null,
  "is_currently_active_DD": true,
  "reason_for_DD_change": "automatic direct debit review",
  "dd_amount": 169.56,
  "dd_amount_change": 11.01,
  "recommended_dd_amount": 162.54,
  "yearly_predicted_energy_cost_gbp": 2034.72,
  "description": "Your DD has been updated.",
  "collectionDay": 2,
  "is_exemption": false,
  "exemption_expiry_date": null,
  "is_amount_manually_reduced_lower_than_recommended_amount": false
}


# Prompt & baseline rollout

The prompt template is the same chat schema the production `direct_debit_explainer` API uses (system message with domain knowledge + user message with the rendered `latest_dd_change` and `account_context`). `dd_explainer.build_chat_messages` renders one for the example above.

In [8]:
messages = build_chat_messages(example)

total_chars = sum(len(m["content"]) for m in messages)
print(f"Chat messages: {len(messages)} ({total_chars} chars total)")
print("---- system (truncated) ----")
print(messages[0]["content"][:400], "...\n")
print("---- user (truncated) ----")
print(messages[1]["content"][:800], "...\n")

Chat messages: 2 (12844 chars total)
---- system (truncated) ----
<task>
Your mission is to provide an explanation to a household energy customer about why their direct debit (DD) has changed (this could be an increase or decrease in their DD amount in £ GBP).
Below you are provided with a list of common triggers for DD changes, and raw data from the customer's account.
You should analyse the triggers and account data together to determine which triggers are rel ...

---- user (truncated) ----
<latest_DD_change>
**Latest DD change:**
{
  "datetime_from": "2026-02-13T08:00:00",
  "datetime_to": null,
  "is_currently_active_DD": true,
  "reason_for_DD_change": "automatic direct debit review",
  "dd_amount": 169.56,
  "dd_amount_change": 11.01,
  "recommended_dd_amount": 162.54,
  "yearly_predicted_energy_cost_gbp": 2034.72,
  "description": "Your DD has been updated.",
  "collectionDay": 2,
  "is_exemption": false,
  "exemption_expiry_date": null,
  "is_amount_manually_reduced_lower_than

First, let's prompt the model without RL and see how it goes:

In [9]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
print("=" * 60)
print("BASE MODEL OUTPUT (before RL training):")
print("=" * 60)

inputs = tokenizer(
    text = text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 1.0, top_p = 0.95, top_k = 64,
)

BASE MODEL OUTPUT (before RL training):
```json
{
  "explanations": [
    {
      "trigger": "Change in usage",
      "header": "Increased Energy Use",
      "explanation": "Your Direct Debit has increased because our system detected a significant rise in your projected electricity consumption. Specifically, the projected annual consumption has increased by 13.95% compared to the last review. This higher expected usage means we have adjusted your payment to better cover your anticipated energy needs."
    },
    {
      "trigger": "Change in unit rates",
      "header": "Unit Rates Increased",
      "explanation": "The change in your Direct Debit amount is partly due to an increase in your energy unit rates. Although your contract rates have not changed recently, the overall cost structure has shifted, leading to a higher recommended payment. This adjustment ensures your payment aligns with the current energy prices."
    },
    {
      "trigger": "Missed/bounced DD payments",
      "h

# Reward functions

Seven verifiable rewards, grouped by purpose:

1. **Shape** — `extract_json` + `reward_schema_valid` + `reward_triggers_in_enum`: output must parse as `DirectDebitExplainerResponse`, and each `trigger` must be a valid enum value.
2. **Correctness** — `reward_triggers_match_ground_truth`: F1 between predicted and oracle trigger sets. This is the main RL signal.
3. **Production failure modes** — `reward_previous_dd_amount_correct`, `reward_no_hallucinated_facts`, `reward_underpayment_language_constrained`: one per category identified in the LangSmith error-analysis reports.
4. **Well-formed explanations** — `reward_explanations_well_formed`: shape/length check.

In [ ]:
from dd_explainer_rewards import (
    extract_json, parse_response, score_completion,
    reward_schema_valid, reward_triggers_in_enum,
    reward_triggers_match_ground_truth,
    reward_previous_dd_amount_correct, reward_no_hallucinated_facts,
    reward_underpayment_language_constrained,
    reward_explanations_well_formed,
    REWARD_FUNCS,
)

# Sanity-check: score a hand-written good response + a bad one. Matches the helper
# lifted into dd_explainer_rewards, so this is identical to running train.py utils.
_good = """```json
{"explanations": [
  {"trigger": "Change in usage",
   "header": "Your usage went up",
   "explanation": "Your projected annual electricity use climbed by 12%, which means more energy to pay for. Your direct debit has been updated so you cover that higher use. Nothing else has changed."}
]}
```"""
_bad = "Here is some text with no JSON."

_fake_inp = example.model_dump(mode="json")
print("good:", score_completion(_good, ["Change in usage"], _fake_inp))
print("bad :", score_completion(_bad,  ["Change in usage"], _fake_inp))

# Dataset Preparation

Create the training dataset.

In [14]:
from pathlib import Path
from datasets import Dataset

DATA_DIR = Path("/workspace/gemma4_rl/data")
jsonl_files = sorted(DATA_DIR.glob("dd_dataset_*_*rows.jsonl"))

if jsonl_files:
    dataset_path = jsonl_files[-1]  # newest by UTC timestamp in filename
    dataset = Dataset.from_json(str(dataset_path))
    # Line 0 is a metadata record marked with "__meta__": True; drop it.
    if "__meta__" in dataset.column_names:
        dataset = dataset.filter(lambda r: r.get("__meta__") is not True).remove_columns("__meta__")
    if "row_index" in dataset.column_names:
        dataset = dataset.remove_columns("row_index")
    print(f"Loaded {len(dataset)} rows from {dataset_path.name}")
    meta_path = dataset_path.with_suffix(".meta.json")
    if meta_path.exists():
        import json as _json
        meta = _json.loads(meta_path.read_text())
        print(f"  generator_version={meta.get('generator_version')}  seed={meta.get('seed')}  generated_at_utc={meta.get('generated_at_utc')}")
else:
    print("No cached JSONL found; generating in memory via build_dataset(n=1000).")
    rows = build_dataset(n=1000, seed=42)
    dataset = Dataset.from_list(rows)

maximum_length = len(tokenizer.apply_chat_template(
    dataset[0]["prompt"],
    add_generation_prompt=True,
))
print(f"Rows: {len(dataset)}")
print(f"Maximum prompt length (tokens): {maximum_length}")
print(f"Sample ground_truth_triggers: {dataset[0]['ground_truth_triggers']}")


Loaded 5500 rows from dd_dataset_20260424T174610Z_5500rows.jsonl
  generator_version=1.0.0  seed=42  generated_at_utc=2026-04-24T17:46:10.083971+00:00
Rows: 5500
Maximum prompt length (tokens): 12895
Sample ground_truth_triggers: ['Change in usage']


<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations! We also support GSPO, GAPO, Dr GRPO and more! Go the Unsloth [Reinforcement Learning Docs](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) for more options.

In [ ]:
# Training dynamics from the previous run flagged policy collapse (KL 18 → 28 → 48 → 406
# over steps 9-12). Diagnosis: lr=5e-5 was too high for Gemma 4 GRPO at temperature=1.0
# with 4+ generations. Fixes: drop lr to 1e-5 and add beta=0.04 to constrain KL drift.
#
# E4B-it is ~2 GB larger than E2B-it. Keeping bs=8 / num_gen=4 so peak VRAM stays
# under ~36 GB with headroom for the FA2 workspace. If VRAM allows after FA2 is
# installed, push to bs=10 num_gen=5 or bs=12 num_gen=6.
#
#   (per_device_bs * grad_accum) must be divisible by num_generations: 8 * 1 = 8 % 4 = 0 ✓.
max_completion_length = 1024

from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 1e-5,                # was 5e-5 — too aggressive, caused KL blowup.
    beta = 0.04,                          # KL penalty; 0.04 is the TRL GRPO default.
    weight_decay = 0.001,
    warmup_steps = 30,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 1,
    num_generations = 4,
    max_completion_length = max_completion_length,
    max_steps = 300,
    save_steps = 50,
    report_to = "none",
    output_dir = "outputs",
    epsilon = 0.2,
    epsilon_high = 0.28,
    delta = 1.5,
    loss_type = 'bnpo',
    mask_truncated_completions = True,
)

And let's run the trainer! Watch the `reward` column — the main component is `reward_triggers_match_ground_truth` (range ≈ -2 to +10). Early on you'll see `reward_schema_valid` punished (the base Gemma 4 E2B rarely produces valid JSON on first try); the shape rewards lift as the model learns the output contract.

Rough expectations on A100 40GB:
- Steps 1-30: shape rewards dominate (schema_valid -2 → +1, explanations_well_formed oscillates). Trigger F1 ≈ 0.
- Steps 30-150: shape stabilises, F1 starts moving.
- Steps 150-300: F1 climbs toward 0.6-0.8; production-failure rewards (previous_amount, no_hallucinated_facts, underpayment) transition from mostly neutral to positive.

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = REWARD_FUNCS,   # from dd_explainer_rewards (7 verifiable rewards)
    args = training_args,
    train_dataset = dataset,
)

And let's train the model!

On A100 40GB with the settings above, expect ~2–3 seconds per step early on (before completions get long), and the reward signal to start rising somewhere after step ~100. If you hit OOM, lower `per_device_train_batch_size` and `num_generations` to 2.

In [17]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,500 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 12 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (12 x 1 x 1) = 12
 "-____-"     Trainable parameters = 119,439,360 of 5,223,736,864 (2.29% trained)
[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_schema_valid / mean,rewards / reward_schema_valid / std,rewards / reward_triggers_in_enum / mean,rewards / reward_triggers_in_enum / std,rewards / reward_triggers_match_ground_truth / mean,rewards / reward_triggers_match_ground_truth / std,rewards / reward_previous_dd_amount_correct / mean,rewards / reward_previous_dd_amount_correct / std,rewards / reward_no_hallucinated_facts / mean,rewards / reward_no_hallucinated_facts / std,rewards / reward_underpayment_language_constrained / mean,rewards / reward_underpayment_language_constrained / std,rewards / reward_explanations_well_formed / mean,rewards / reward_explanations_well_formed / std
1,-0.137891,3.583333,5.247655,189.333344,1.000000,301.000000,0.000000,189.333344,1.000000,301.000000,-0.000000,0.500000,1.167748,0.666667,0.778499,1.333333,3.550502,0.000000,0.000000,0.833333,0.389249,0.500000,0.000000,-0.250000,0.452267
2,-0.064800,6.750000,4.002840,193.750000,1.000000,308.000000,0.000000,193.750000,1.000000,308.000000,-0.000000,0.750000,0.866025,0.833333,0.577350,6.000000,3.302891,0.000000,0.000000,-0.750000,2.005674,0.333333,0.577350,-0.416667,0.288675
3,-0.150095,3.883333,4.558675,244.333344,1.000000,850.000000,0.000000,244.333344,1.000000,850.000000,0.009887,0.500000,1.167748,0.666667,0.778499,5.466667,3.803667,-0.250000,0.866025,-2.500000,1.167748,0.500000,0.000000,-0.500000,0.000000
4,-0.125397,4.500000,3.205110,177.666672,1.000000,211.000000,0.000000,177.666672,1.000000,211.000000,0.008499,0.750000,0.866025,0.833333,0.577350,5.666667,2.674232,0.000000,0.000000,-2.750000,0.866025,0.500000,0.000000,-0.500000,0.000000
5,-0.148188,1.750000,4.454314,205.750000,1.000000,289.000000,0.000000,205.750000,1.000000,289.000000,0.134185,0.500000,1.167748,0.666667,0.778499,-0.333333,3.055051,0.000000,0.000000,0.833333,0.389249,0.500000,0.000000,-0.416667,0.288675
6,-0.281599,1.216667,3.589336,177.250000,1.000000,277.000000,0.000000,177.250000,1.000000,277.000000,0.652984,0.500000,1.167748,0.666667,0.778499,-1.200000,1.868397,0.000000,0.000000,0.833333,0.389249,0.500000,0.000000,-0.083333,0.514929
7,0.003549,1.833333,1.696699,207.500000,187.000000,275.000000,0.000000,207.500000,187.000000,275.000000,0.001767,1.000000,0.000000,1.000000,0.000000,-1.500000,1.732051,0.000000,0.000000,1.000000,0.000000,0.500000,0.000000,-0.166667,0.492366
8,0.235780,-0.333333,2.839121,238.000000,1.000000,1024.000000,0.083333,166.545456,1.000000,229.000000,1.939454,0.250000,1.356801,0.500000,0.904534,-2.000000,0.000000,0.000000,0.000000,0.750000,0.452267,0.500000,0.000000,-0.333333,0.389249
9,-0.290799,0.583333,3.752777,152.583344,1.000000,217.000000,0.000000,152.583344,1.000000,217.000000,18.420620,0.250000,1.356801,0.500000,0.904534,0.000000,3.618136,0.000000,0.000000,-0.250000,1.712255,0.500000,0.000000,-0.416667,0.288675
10,-0.187805,0.000000,3.045115,143.250000,1.000000,202.000000,0.000000,143.250000,1.000000,202.000000,28.251286,0.250000,1.356801,0.500000,0.904534,-2.000000,0.000000,0.000000,0.000000,0.750000,0.452267,0.500000,0.000000,0.000000,0.522233


KeyboardInterrupt: 

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_pretrained("gemma_4_lora")  # Local saving
tokenizer.save_pretrained("gemma_4_lora")

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("gemma_4_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

# Regression against real failure traces

The LangSmith online evaluator flagged 187 prompts in one 7-day window (35% of 704) as faithfulness failures. We have them in `.error_analysis_cache/20260413T075447Z_20260420T075447Z/traces.parquet`. Re-score both the base and GRPO-tuned model on those same prompts using our reward functions — the headline number is how many now pass `reward_previous_dd_amount_correct` and `reward_no_hallucinated_facts`.

In [ ]:
import pandas as pd, torch
from pathlib import Path
from train import reconstruct_pin_from_trace, prompt_token_length

trace_dir = Path("/workspace/gemma4_rl/.error_analysis_cache/20260413T075447Z_20260420T075447Z")
df = pd.read_parquet(trace_dir / "traces.parquet")
failed = df[df["feedback.direct_debit_faithfulness"] < 1.0].copy()
print(f"Loaded {len(failed)} previously-failed prompts")

budget = max_seq_length - max_completion_length - 64
candidates = []
for _, row in failed.iterrows():
    pin = reconstruct_pin_from_trace(row)
    if pin is None:
        continue
    if prompt_token_length(tokenizer, pin) <= budget:
        candidates.append(pin)
print(f"Candidates after token-length filter: {len(candidates)} / {len(failed)}")


@torch.no_grad()
def _generate(pin) -> str:
    encoded = tokenizer.apply_chat_template(
        build_chat_messages(pin),
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    ).to("cuda")
    out = model.generate(**encoded, max_new_tokens=max_completion_length,
                         temperature=0.0, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][encoded["input_ids"].shape[1]:], skip_special_tokens=True)


# Subsample to 20 rows for a fast regression pass; raise for a fuller view.
N_REGRESS = 20
rows_out = []
for pin in candidates[:N_REGRESS]:
    completion = _generate(pin)
    gt = sorted(t.value for t in detect_triggers(pin))
    rows_out.append(score_completion(completion, gt, pin.model_dump(mode="json")))

regression_df = pd.DataFrame(rows_out)
print(f"\nScored {len(regression_df)} rows on the tuned model. Mean reward by category:")
print(regression_df.mean(numeric_only=True).round(3).to_string())

<a name="Inference"></a>
# Inference
Now let's try the model we just trained!

In [ ]:
# Roll out the tuned model on a fresh DD example and confirm the output parses.
import random
rng_demo = random.Random(2026)
demo_target = {Trigger.Change_in_unit_rates}
demo_example = generate_dd_example(demo_target, rng_demo)

text = tokenizer.apply_chat_template(
    build_chat_messages(demo_example),
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(images=None, text=text, add_special_tokens=False, return_tensors="pt").to("cuda"),
    temperature = 0.0,
    do_sample = False,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
    use_cache = True,
)

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("gemma_4_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/gemma_4_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("gemma_4_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/gemma_4_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("gemma_4_lora")
    tokenizer.save_pretrained("gemma_4_lora")
if False:
    model.push_to_hub("HF_USERNAME/gemma_4_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/gemma_4_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("gemma_4_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/gemma_4_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("gemma_4_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/gemma_4_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("gemma_4_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/gemma_4_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/gemma_4_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Now, use the `gemma_4_finetune.Q8_0.gguf` file or `gemma_4_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).